In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../dados/simulados/respostas_simuladas.csv")

In [ ]:
# Variáveis selecionadas para o clustering
#
# Foco em duas dimensões principais:
#   1. Perfil do gerador: tamanho e tipo da unidade
#      (tipo_gerador, quantidade_semanal, num_pessoas)
#   2. Comportamento e potencial de mudança em relação aos resíduos
#      (destino_atual, interesse_aprender, tentativa_reutilizacao)
#
# Variáveis como tipo_atividade, bairro, tipo_residuo_principal,
# residuo_organico_principal, origem_residuo, caracteristica_residuo,
# custo_destinacao e potencial_reaproveitamento foram mantidas de fora
# nesta versão para manter o modelo simples e explicável no MVP.
# Elas podem ser incorporadas em versões futuras para refinar os perfis.

colunas = [
    "tipo_gerador",
    "quantidade_semanal",
    "num_pessoas",
    "destino_atual",
    "interesse_aprender",
    "tentativa_reutilizacao",
]

In [4]:
df_cluster = df[colunas].copy()
df_cluster.head()

,tipo_gerador,quantidade_semanal,num_pessoas,destino_atual,interesse_aprender,tentativa_reutilizacao
0,Pessoa física (residência),Não sei estimar,1 a 3,Coleta pública comum,Sim,Não
1,Pessoa física (residência),1 a 5 kg por semana,10 a 30,Coleta pública comum,Sim,Não
2,Instituição pública,5 a 10 kg por semana,4 a 6,Doação,Sim,Sim
3,Pessoa física (residência),5 a 10 kg por semana,4 a 6,Doação,Sim,Não
4,Pessoa física (residência),5 a 10 kg por semana,10 a 30,Doação,Sim,Não


In [5]:
# One-hot encoding das colunas categóricas
X = pd.get_dummies(df_cluster, drop_first=False)

X.head()
X.shape

(150, 26)

In [6]:
k = 3 
seed = 2016

kmeans = KMeans(n_clusters=k, random_state=seed, n_init='auto')

df_cluster["cluster"] = kmeans.fit_predict(X)
df_cluster["cluster"].value_counts().sort_index()


cluster
0    37
1    37
2    76
Name: count, dtype: int64

In [ ]:
# Rótulos interpretáveis para cada cluster
#
# Cluster 0 — "Engajados": já tentaram reutilizar e querem aprender mais.
# Cluster 1 — "Indecisos": não tentaram e estão em dúvida se querem aprender.
# Cluster 2 — "Potencial latente": querem aprender, mas nunca tentaram.

rotulos = {0: "Engajados", 1: "Indecisos", 2: "Potencial latente"}
df_cluster["perfil"] = df_cluster["cluster"].map(rotulos)
df_cluster["perfil"].value_counts()

In [ ]:
df_cluster.groupby("perfil")[[
    "tipo_gerador",
    "destino_atual",
    "interesse_aprender",
    "tentativa_reutilizacao",
]].agg(pd.Series.mode)

In [ ]:
cores_perfil = {"Engajados": "tab:blue", "Indecisos": "tab:orange", "Potencial latente": "tab:green"}
contagens = df_cluster["perfil"].value_counts().reindex(cores_perfil.keys())

plt.figure(figsize=(6, 4))
contagens.plot(kind="bar", color=[cores_perfil[p] for p in contagens.index])
plt.title("Distribuição dos perfis identificados")
plt.xlabel("Perfil")
plt.ylabel("Quantidade de respondentes")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Redução para 2 dimensões com PCA para visualização
pca = PCA(n_components=2, random_state=2026)
componentes = pca.fit_transform(X)

plt.figure(figsize=(7, 5))
for cluster_id, nome in rotulos.items():
    mask = df_cluster["cluster"] == cluster_id
    plt.scatter(
        componentes[mask, 0],
        componentes[mask, 1],
        label=nome,
        alpha=0.7,
        edgecolors="w",
        linewidth=0.5,
    )

plt.title("Perfis de geradores — projeção PCA (2 componentes)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} da variância)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} da variância)")
plt.legend(title="Perfil")
plt.tight_layout()
plt.show()

In [ ]:
df_potencial = df_cluster[df_cluster["perfil"] == "Potencial latente"]

print("Interesse em aprender:")
print(df_potencial["interesse_aprender"].value_counts(normalize=True))
print()
print("Tentativa de reutilização:")
print(df_potencial["tentativa_reutilizacao"].value_counts(normalize=True))

In [ ]:
tab = pd.crosstab(df_cluster["perfil"], df_cluster["interesse_aprender"], normalize="index")
tab = tab.reindex(["Engajados", "Indecisos", "Potencial latente"])

tab.plot(kind="bar", stacked=True, figsize=(7, 4), colormap="viridis")
plt.title("Interesse em aprender por perfil")
plt.xlabel("Perfil")
plt.ylabel("Proporção dentro do perfil")
plt.legend(title="Interesse em aprender", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Limitações do protótipo

Este notebook é um MVP exploratório e possui limitações conhecidas: a amostra é simulada e pequena (n=150), apenas 6 das 14 variáveis disponíveis foram utilizadas no modelo, e o número de clusters (k=3) foi uma escolha exploratória, sem validação formal por métricas como silhouette score ou método do cotovelo. Os resultados servem como ponto de partida para a análise com dados reais e para a definição de uma metodologia mais robusta nas próximas fases do projeto.